# Manual Holt-Winters Model

## Purpose
Fits a manually specified **Holt-Winters (additive trend, damped, additive seasonal)** model using walk-forward validation as an initial proof-of-concept. The configuration is set directly based on domain knowledge rather than a parameter search.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
- The walk-forward predictions and RMSE.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from math import sqrt

## Load Training Data

In [2]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.index.freq = pd.infer_freq(series.index)
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Freq: MS, Name: Sales, dtype: int64

## Train / Test Split

50/50 split, consistent with the persistence baseline notebook, enabling a direct RMSE comparison.

In [3]:
# prepare data
X = series.values.astype('float64')
train_size = int(len(X) * 0.50)
train, test = X[:train_size], X[train_size:]

## Model Configuration

Parameters are fixed manually for this proof-of-concept run. `trend='add'` with `damped_trend=True` applies a linear trend component that decays toward a flat projection, preventing over-extrapolation. `seasonal='add'` is appropriate when seasonal amplitude is roughly constant across the series. `seasonal_periods=12` matches the monthly data.

In [4]:
# Holt-Winters configuration
# trend='add'      : linear trend component (additive)
# damped_trend=True: damps the trend projection to prevent over-extrapolation
# seasonal='add'   : seasonal component (additive; appropriate when seasonal
#                    amplitude is roughly constant across the series)
# seasonal_periods : 12 months
HW_TREND    = 'add'
HW_DAMPED   = True
HW_SEASONAL = 'add'
HW_PERIODS  = 12

## Fit Validity Guard

`ExponentialSmoothing` does not expose an `mle_retvals` convergence flag like ARIMA. The equivalent guard is to verify that all estimated smoothing parameters are finite and within `[0, 1]` — values outside this range signal a degenerate or failed optimisation. Bounds are closed: `smoothing_trend=0.0` is a valid outcome indicating the optimiser found no per-step trend update is needed, not a failure.

In [5]:
def is_hw_fit_valid(fit):
    """Return True if all smoothing params are finite and within [0, 1].

    ExponentialSmoothing does not expose mle_retvals like ARIMA; checking
    that the estimated smoothing weights are bounded is the equivalent
    empirical guard against a failed/degenerate optimisation.

    Bounds are closed [0, 1]: a value of exactly 0 for smoothing_trend is
    valid — it means the optimiser found no per-step trend update is needed
    (trend stays at its initial estimate). Rejecting 0.0 with a strict
    lower bound would incorrectly flag this legitimate outcome as a failure.
    The true failure signals are NaN, inf, or values outside [0, 1].
    """
    param_keys = ['smoothing_level', 'smoothing_trend', 'smoothing_seasonal']
    for key in param_keys:
        val = fit.params.get(key)
        if val is None:
            continue
        if not np.isfinite(val) or not (0.0 <= val <= 1.0):
            return False
    return True

## Walk-Forward Validation

At each step the model is re-fit from scratch on the current history window and used to forecast one step ahead. The true observation is then appended to history before advancing to the next step — ensuring no look-ahead bias. The validity guard is applied after every fit; an invalid parameter set raises a `RuntimeError` immediately.

In [6]:
# walk-forward validation
history = list(train)
predictions = []

for i in range(len(test)):
    model = ExponentialSmoothing(
        history,
        trend=HW_TREND,
        damped_trend=HW_DAMPED,
        seasonal=HW_SEASONAL,
        seasonal_periods=HW_PERIODS,
        initialization_method='estimated'
    )
    fit = model.fit(optimized=True)

    if not is_hw_fit_valid(fit):
        raise RuntimeError(
            f"Holt-Winters did not produce valid smoothing parameters at step {i}: "
            f"{fit.params}"
        )

    yhat = fit.forecast(1)[0]
    predictions.append(yhat)
    obs = test[i]
    history.append(obs)
    print(f'>Predicted={yhat:.3f}, Expected={obs:.3f}')

>Predicted=7721.427, Expected=8314.000
>Predicted=9376.057, Expected=10651.000
>Predicted=4758.724, Expected=3633.000
>Predicted=3920.937, Expected=4292.000
>Predicted=4500.526, Expected=4154.000
>Predicted=4624.535, Expected=4121.000
>Predicted=4973.715, Expected=4647.000
>Predicted=4777.800, Expected=4753.000
>Predicted=4153.853, Expected=3965.000
>Predicted=2988.639, Expected=1723.000
>Predicted=4585.447, Expected=5048.000
>Predicted=5821.927, Expected=6922.000
>Predicted=8101.948, Expected=9858.000
>Predicted=10294.959, Expected=11331.000
>Predicted=4889.124, Expected=4016.000
>Predicted=4470.439, Expected=3957.000
>Predicted=4924.072, Expected=4510.000
>Predicted=4896.348, Expected=4276.000
>Predicted=5191.414, Expected=4968.000
>Predicted=5162.975, Expected=4677.000
>Predicted=4470.882, Expected=3523.000
>Predicted=3099.673, Expected=1821.000
>Predicted=5262.263, Expected=5222.000
>Predicted=6728.858, Expected=6872.000
>Predicted=9532.573, Expected=10803.000
>Predicted=11443.474,

## Report RMSE

Compare this RMSE against the persistence baseline. An improvement confirms that Holt-Winters structure adds predictive value over the naïve model.

In [7]:
# report performance
rmse = sqrt(mean_squared_error(test, predictions))
print('RMSE: %.3f' % rmse)

RMSE: 969.634
